# HT-9046MX Shared LSTM Autoencoder (Google Colab)

Notebook นี้เทรน **LSTM Autoencoder กลาง 1 โมเดล** จากหลาย handler/module แต่เก็บ **scaler และ anomaly threshold แยกตาม `machine_id + module_id`** เพื่อไม่ให้ความต่างของ baseline แต่ละเครื่องถูกตีความเป็น anomaly

## Goal

- ใช้เฉพาะช่วง `Status_n=On`, `Busy_n=0`
- ตัด `ChangeValve`, `AdjustValve`, `MValveHome`, Valve ติดลบ และค่าอุณหภูมิ sentinel `-200`
- ไม่สร้าง window ข้าม transition หรือช่องว่างเวลา
- เทรน shared model ด้วยข้อมูลที่ normalize แยกแต่ละ machine-module
- calibrate threshold และรายงาน test metrics แยกแต่ละ machine-module
- export model, scalers, thresholds, metrics และ inference CSV ไปยัง Google Drive


## Setup

1. ใน Colab เลือก **Runtime → Change runtime type → T4 GPU**
2. ค่าเริ่มต้น `USE_DRIVE = False`: อัปโหลด `ht9046mx_colab_full_package.zip` เข้า `/content` ผ่าน Files ของ Colab
3. Package มีเฉพาะ source code และ prepared dataset `shared_full_v2`; raw logs ไม่ขึ้น GitHub หรือ Google Drive
4. Notebook จะแตก package เข้า `/content/ht9046mx_runtime_full_v2` และดาวน์โหลด Runtime ZIP อัตโนมัติเมื่อเทรนเสร็จ
5. หาก Drive mount ใช้งานได้ ให้เปลี่ยน `USE_DRIVE = True` เพื่ออ่าน/บันทึกผ่าน `MyDrive/Data Analysis`
6. Notebook ชุดนี้ตั้ง `RUN_MODE = "full"` และเทรนสูงสุด 30 epochs พร้อม EarlyStopping


In [ ]:
# @title 1. Set parameters and prepare runtime
from pathlib import Path
import os
import shutil
import hashlib

IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
USE_DRIVE = False  # True = persist directly to MyDrive; False = upload package to /content
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/Data Analysis')
elif IN_COLAB:
    DRIVE_PROJECT_DIR = Path('/content/Data Analysis')
    DRIVE_PROJECT_DIR.mkdir(parents=True, exist_ok=True)
else:
    DRIVE_PROJECT_DIR = Path.cwd()

RUN_MODE = 'full'  # 'smoke' or 'full'
PREPARED_DATASET_NAME = 'shared_smoke_v2' if RUN_MODE == 'smoke' else 'shared_full_v2'
PACKAGE_ROOT = DRIVE_PROJECT_DIR if (USE_DRIVE or not IN_COLAB) else Path('/content')
PACKAGE_ZIP = PACKAGE_ROOT / 'ht9046mx_colab_full_package.zip'
EXPECTED_PACKAGE_SHA256 = '14b8b197c572b20134250b10a1f354202a4556952aa630edd84cc1ab32f1fd55'
if IN_COLAB and not USE_DRIVE:
    package_parts = sorted(Path('/content').glob('ht9046mx_colab_full_package.zip.part.*'))
    if package_parts:
        combined_zip = Path('/content/ht9046mx_colab_full_package.zip.complete')
        with combined_zip.open('wb') as output_handle:
            for part_path in package_parts:
                with part_path.open('rb') as input_handle:
                    shutil.copyfileobj(input_handle, output_handle, length=8 * 1024 * 1024)
        os.replace(combined_zip, PACKAGE_ZIP)
        print('Combined package parts:', [path.name for path in package_parts])
if IN_COLAB:
    assert PACKAGE_ZIP.exists(), f'Package not found: {PACKAGE_ZIP}'
    package_hasher = hashlib.sha256()
    with PACKAGE_ZIP.open('rb') as package_handle:
        for package_block in iter(lambda: package_handle.read(8 * 1024 * 1024), b''):
            package_hasher.update(package_block)
    package_sha256 = package_hasher.hexdigest()
    assert package_sha256 == EXPECTED_PACKAGE_SHA256, {
        'expected_sha256': EXPECTED_PACKAGE_SHA256, 'actual_sha256': package_sha256,
    }
    print('Package SHA-256 verified:', package_sha256)
    PROJECT_DIR = Path('/content/ht9046mx_runtime_full_v2')
    expected_bundle = PROJECT_DIR / 'prepared_dataset' / PREPARED_DATASET_NAME
    adaptive_code = PROJECT_DIR / 'compressor_ml' / 'adaptive.py'
    if not expected_bundle.exists() or not adaptive_code.exists():
        PROJECT_DIR.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(PACKAGE_ZIP, PROJECT_DIR)
else:
    PROJECT_DIR = Path(os.environ.get('HT9046_PROJECT_DIR', str(Path.cwd())))
PREPARED_DATASET_DIR = PROJECT_DIR / 'prepared_dataset' / PREPARED_DATASET_NAME
ARTIFACT_DIR = DRIVE_PROJECT_DIR / 'artifacts' / f'shared_lstm_colab_{RUN_MODE}'

EPOCHS = 2 if RUN_MODE == 'smoke' else 30
BATCH_SIZE = 128
MIN_WINDOWS_PER_GROUP = 30
RANDOM_SEED = 42

assert RUN_MODE in {'smoke', 'full'}
assert PROJECT_DIR.exists(), f'PROJECT_DIR not found: {PROJECT_DIR}'
assert PREPARED_DATASET_DIR.exists(), f'Prepared dataset not found: {PREPARED_DATASET_DIR}'
print('Runtime project:', PROJECT_DIR)
print('Output project:', DRIVE_PROJECT_DIR)
print('Persistence:', 'Google Drive' if USE_DRIVE else 'Colab runtime + automatic download')
print('Prepared dataset:', PREPARED_DATASET_DIR)
print('Artifacts:', ARTIFACT_DIR)
print('Mode:', RUN_MODE)


In [ ]:
# @title 2. Import project code and initialize runtime
import json
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
try:
    from IPython.display import display
except ImportError:
    display = print

sys.path.insert(0, str(PROJECT_DIR))
from compressor_ml.anomaly import (
    StandardScaler3D, anomaly_score, health_score, pseudo_label,
    reconstruction_error,
)
from compressor_ml.model import build_lstm_autoencoder
from compressor_ml.prepare_dataset import load_prepared_dataset, safe_group_name

np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## Context & Methods

### Key assumptions

- ข้อมูลไม่มี fault label จึงเป็น unsupervised anomaly detection ไม่ใช่การวินิจฉัย root cause หรือ RUL
- baseline ปกติของแต่ละ machine-module ต่างกัน จึง fit scaler เฉพาะจาก train split ของ group นั้น
- shared model เห็นข้อมูล normalized จากทุก group แต่ validation threshold ยังคงแยกต่อ group
- split 70/15/15 ตามเวลา **ภายในแต่ละ group** เพื่อป้องกันข้อมูลอนาคตรั่วเข้า train
- sample weight ทำให้แต่ละ group มีอิทธิพลรวมใกล้เคียงกัน แม้จำนวน windows ต่างกัน


In [ ]:
# @title 3. Load the prepared, unscaled dataset bundle
CONFIG, DATASET_MANIFEST, group_datasets = load_prepared_dataset(PREPARED_DATASET_DIR)
CONFIG.epochs = EPOCHS
CONFIG.batch_size = BATCH_SIZE
CONFIG.validate()
quality_df = pd.read_csv(PREPARED_DATASET_DIR / 'data_quality_summary.csv')

bundle_rows = []
for group in DATASET_MANIFEST['groups']:
    bundle_rows.append({
        'machine_id': group['machine_id'],
        'module_id': group['module_id'],
        **group['windows'],
    })
bundle_table = pd.DataFrame(bundle_rows).sort_values(['machine_id', 'module_id']).reset_index(drop=True)
display(bundle_table)
print('Dataset version:', DATASET_MANIFEST['dataset_version'])
print('Features:', len(CONFIG.feature_columns), '| Window rows:', CONFIG.window_rows)
if len(group_datasets) < 2:
    raise ValueError('Shared model needs at least two prepared machine-module groups.')


## Data preparation

Raw parsing, state filtering และ feature engineering ถูกทำไว้ใน prepared bundle แล้ว แต่ arrays ยังไม่ถูก scale เพื่อให้ cell ถัดไป fit scaler จาก train partition ของแต่ละ machine-module เท่านั้น


In [ ]:
# @title 4. Fit train-only scalers for every prepared group
group_scalers = {}
for group_key, dataset in group_datasets.items():
    scaler = StandardScaler3D().fit(dataset['train'])
    group_scalers[group_key] = scaler
    dataset['train'] = scaler.transform(dataset['train'])
    dataset['validation'] = scaler.transform(dataset['validation'])
    dataset['test'] = scaler.transform(dataset['test'])

print('Train-only scalers fitted:', len(group_scalers))
display(quality_df.groupby(['machine_id', 'module_id'], as_index=False)[
    ['accepted_rows', 'rejected_rows', 'complete_windows', 'selected_windows']
].sum().head(20))


In [ ]:
# @title 5. Pool normalized windows with balanced group weights
train_parts = []
validation_parts = []
weight_parts = []

for group_key, dataset in group_datasets.items():
    group_train = dataset['train']
    train_parts.append(group_train)
    validation_parts.append(dataset['validation'])
    weight_parts.append(np.full((len(group_train), CONFIG.window_rows), 1.0 / len(group_train), dtype=np.float32))

pooled_train = np.concatenate(train_parts).astype(np.float32)
pooled_validation = np.concatenate(validation_parts).astype(np.float32)
sample_weights = np.concatenate(weight_parts)
sample_weights *= sample_weights.size / sample_weights.sum()

print('Valid groups:', len(group_datasets))
print('Pooled train:', pooled_train.shape)
print('Pooled validation:', pooled_validation.shape)
print('Sample-weight mean:', float(sample_weights.mean()))


## Shared model training

โมเดลส่วนกลางจะเรียนรู้รูปแบบ temporal ที่ใช้ร่วมกันจากข้อมูล normalized ทุก group ส่วน baseline เชิงตัวเลขยังถูกเก็บไว้ใน scaler ของแต่ละ group


In [ ]:
# @title 6. Train one shared LSTM Autoencoder
tf.keras.backend.clear_session()
shared_model = build_lstm_autoencoder(CONFIG, pooled_train.shape[-1])
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )
]
history = shared_model.fit(
    pooled_train, pooled_train,
    sample_weight=sample_weights,
    validation_data=(pooled_validation, pooled_validation),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    callbacks=callbacks,
    verbose=2,
)

pd.DataFrame(history.history).plot(figsize=(8, 4), title='Shared model training history')
plt.xlabel('Epoch')
plt.ylabel('MAE loss')
plt.grid(alpha=0.25)
plt.show()


In [ ]:
# @title 7. Calibrate a threshold and evaluate each group
group_thresholds = {}
metric_rows = []

for (machine_id, module_id), dataset in group_datasets.items():
    validation_pred = shared_model.predict(dataset['validation'], batch_size=BATCH_SIZE, verbose=0)
    validation_errors, _ = reconstruction_error(dataset['validation'], validation_pred)
    threshold = float(np.percentile(validation_errors, CONFIG.threshold_percentile))

    test_pred = shared_model.predict(dataset['test'], batch_size=BATCH_SIZE, verbose=0)
    test_errors, _ = reconstruction_error(dataset['test'], test_pred)
    safe_name = safe_group_name(machine_id, module_id)
    group_thresholds[safe_name] = {
        'machine_id': machine_id, 'module_id': module_id,
        'percentile': CONFIG.threshold_percentile, 'value': threshold,
    }
    metric_rows.append({
        'machine_id': machine_id, 'module_id': module_id,
        'validation_mae_p50': float(np.median(validation_errors)),
        'validation_mae_p99': threshold,
        'test_mae_p50': float(np.median(test_errors)),
        'test_mae_p95': float(np.percentile(test_errors, 95)),
        'test_exceedance_rate': float(np.mean(test_errors > threshold)),
    })

metrics_df = pd.DataFrame(metric_rows).sort_values(['machine_id', 'module_id']).reset_index(drop=True)
display(metrics_df.round(4))

ax = metrics_df.pivot(index='machine_id', columns='module_id', values='test_exceedance_rate').plot(
    kind='bar', figsize=(11, 4), title='Test threshold exceedance rate by machine-module'
)
ax.set_ylabel('Fraction above validation p99 threshold')
ax.axhline(0.10, color='red', linestyle='--', linewidth=1, label='Review level (10%)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# @title 8. Save the shared model and per-group calibration bundle
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
scaler_dir = ARTIFACT_DIR / 'scalers'
scaler_dir.mkdir(parents=True, exist_ok=True)

shared_model.save(ARTIFACT_DIR / 'shared_model.keras')
CONFIG.save(ARTIFACT_DIR / 'config.json')
for (machine_id, module_id), scaler in group_scalers.items():
    scaler.save(str(scaler_dir / f'{safe_group_name(machine_id, module_id)}.npz'))

with (ARTIFACT_DIR / 'thresholds.json').open('w', encoding='utf-8') as handle:
    json.dump(group_thresholds, handle, indent=2, ensure_ascii=False)
metrics_df.to_csv(ARTIFACT_DIR / 'group_metrics.csv', index=False)
quality_df.to_csv(ARTIFACT_DIR / 'data_quality_summary.csv', index=False)

manifest = {
    'model_type': 'shared_lstm_autoencoder_with_per_group_calibration',
    'model_version': f'shared_lstm_{RUN_MODE}_v1',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'run_mode': RUN_MODE,
    'groups': [safe_group_name(machine_id, module_id) for machine_id, module_id in group_datasets],
    'prepared_dataset_version': DATASET_MANIFEST['dataset_version'],
    'prepared_dataset_format': DATASET_MANIFEST['format'],
    'source_files': DATASET_MANIFEST['sources'],
    'epochs_completed': len(history.history['loss']),
    'final_loss': float(history.history['loss'][-1]),
    'final_validation_loss': float(history.history['val_loss'][-1]),
    'threshold_method': f'per_group_validation_p{CONFIG.threshold_percentile:g}',
}
with (ARTIFACT_DIR / 'manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(manifest, handle, indent=2, ensure_ascii=False)

print('Saved artifact bundle to:', ARTIFACT_DIR)
print(sorted(path.name for path in ARTIFACT_DIR.iterdir()))


## Smoke inference on MX007

cell ต่อไปให้คะแนน test partition ที่ไม่เคยเข้า train ของ `MX007 / Module 1` ด้วย shared model, scaler และ threshold ของ group นั้น


In [ ]:
# @title 9. Score a held-out prepared test partition
SCORE_MACHINE = 'MX007'
SCORE_MODULE = 1

def score_prepared_group(machine_id: str, module_id: int) -> pd.DataFrame:
    group_key = (machine_id, module_id)
    if group_key not in group_scalers:
        raise KeyError(f'No scaler/threshold available for {group_key}.')
    group_info = next(
        group for group in DATASET_MANIFEST['groups']
        if group['machine_id'] == machine_id and int(group['module_id']) == module_id
    )
    metadata = pd.read_csv(PREPARED_DATASET_DIR / group_info['metadata_file'])
    window_meta = metadata.loc[metadata['split'].eq('test')].sort_values('split_index').reset_index(drop=True)
    scaled = group_datasets[group_key]['test']
    if len(scaled) != len(window_meta):
        raise ValueError('Prepared test array and metadata length do not match.')
    reconstructed = shared_model.predict(scaled, batch_size=BATCH_SIZE, verbose=0)
    errors, feature_errors = reconstruction_error(scaled, reconstructed)
    threshold = group_thresholds[safe_group_name(machine_id, module_id)]['value']
    scores = anomaly_score(errors, threshold)

    result = window_meta.copy()
    result['reconstruction_error'] = errors
    result['anomaly_score'] = scores
    result['health_score'] = result.groupby(
        ['machine_id', 'module_id', 'segment_id'], sort=False
    )['anomaly_score'].transform(
        lambda values: health_score(values.to_numpy(), CONFIG.health_smoothing_windows)
    )
    result['condition_status'] = [
        pseudo_label(value, CONFIG.normal_min, CONFIG.watch_min, CONFIG.warning_min)
        for value in result['health_score']
    ]
    top_indices = feature_errors.argmax(axis=1)
    result['top_error_feature'] = [CONFIG.feature_columns[index] for index in top_indices]
    result['threshold'] = threshold
    result['model_version'] = manifest['model_version']
    return result

inference_df = score_prepared_group(SCORE_MACHINE, SCORE_MODULE)
inference_path = ARTIFACT_DIR / f'inference_{safe_group_name(SCORE_MACHINE, SCORE_MODULE)}.csv'
inference_df.to_csv(inference_path, index=False)

latest_columns = [
    'timestamp', 'reconstruction_error', 'anomaly_score', 'health_score',
    'condition_status', 'top_error_feature', 'threshold', 'model_version',
]
display(inference_df[latest_columns].tail(10))
print('Latest result:', inference_df[latest_columns].iloc[-1].to_dict())
print('Saved:', inference_path)


In [ ]:
# @title 10. Visual check of MX_007 health score
plot_frame = inference_df.tail(300).copy()
plot_frame['timestamp'] = pd.to_datetime(plot_frame['timestamp'])
ax = plot_frame.plot(
    x='timestamp', y='health_score', figsize=(12, 4),
    title=f'{SCORE_MACHINE} Module {SCORE_MODULE} — smoothed health score',
    legend=False,
)
for level, label, color in [
    (CONFIG.normal_min, 'Normal', 'green'),
    (CONFIG.watch_min, 'Watch', 'orange'),
    (CONFIG.warning_min, 'Warning', 'red'),
]:
    ax.axhline(level, linestyle='--', linewidth=1, color=color, label=label)
ax.set_ylim(0, 100)
ax.set_ylabel('Health score')
ax.legend()
plt.tight_layout()
plt.show()


## Checks & Next Steps

ก่อนเปลี่ยนเป็น full training ให้ตรวจสิ่งต่อไปนี้:

1. `quality_df` ต้องมี windows จากหลายเครื่องและไม่มีเครื่องใดหายไปโดยไม่ทราบสาเหตุ
2. ดู `test_exceedance_rate` แยก group; ค่าสูงกว่า 10% ควรตรวจ distribution shift, maintenance event หรือ baseline ที่ปน anomaly
3. Smoke result ใช้ยืนยันว่า pipeline รันครบเท่านั้น ห้ามใช้ตัดสินสภาพเครื่องหรือกำหนด maintenance action
4. Full run ควรใช้หลายวันต่อเครื่อง, `RUN_MODE='full'`, และตรวจผลแยกทุก machine-module
5. `condition_status` เป็น pseudo-label จาก reconstruction error ไม่ใช่ fault diagnosis และไม่ใช่ RUL
6. หากเพิ่มเครื่องใหม่ ให้เก็บ normal baseline เพื่อ fit scaler/threshold ใหม่ โดยยังใช้ shared model เดิมได้หลังผ่าน validation


## Adaptive System Bootstrap

ส่วนนี้สร้าง Golden Calibration และ Frozen Holdout แยกทุก `machine_id + module_id` หลัง Shared Model เทรนเสร็จ จากนั้นบันทึก Adaptive Seed และ ZIP สำหรับนำไปใช้กับ Automatic Scoring บน Windows

หลักความปลอดภัย:

- Shared Model และ train-only scaler จะไม่ถูกปรับอัตโนมัติ
- Golden Profile จะไม่ถูกเขียนทับ
- ระบบปรับเฉพาะ operational calibration หลังผ่าน validation และ shadow observations
- การ validate นี้เป็น unsupervised calibration validation ไม่ใช่การยืนยัน fault accuracy

In [ ]:
# @title 11. Build immutable golden profiles and adaptive seed
from compressor_ml.adaptive import AdaptiveConfig
from compressor_ml.adaptive_runner import bootstrap_from_prepared

ADAPTIVE_SEED_DIR = ARTIFACT_DIR / 'adaptive_seed'
adaptive_config = AdaptiveConfig.load(PROJECT_DIR / 'configs' / 'adaptive_calibration.json')

# A rerun may replace only this generated seed inside the current artifact.
if ADAPTIVE_SEED_DIR.exists():
    assert ADAPTIVE_SEED_DIR.parent.resolve() == ARTIFACT_DIR.resolve()
    shutil.rmtree(ADAPTIVE_SEED_DIR)

adaptive_seed_summary = bootstrap_from_prepared(
    PREPARED_DATASET_DIR,
    ARTIFACT_DIR,
    ADAPTIVE_SEED_DIR,
    adaptive_config=adaptive_config,
    model=shared_model,
    batch_size=BATCH_SIZE,
)
display(pd.DataFrame([adaptive_seed_summary]))

In [ ]:
# @title 12. Validate seed coverage and create the Windows runtime ZIP
seed_manifest = json.loads((ADAPTIVE_SEED_DIR / 'seed_manifest.json').read_text(encoding='utf-8'))
seed_groups = {row['group_name'] for row in seed_manifest['groups']}
artifact_groups = set(group_thresholds)

assert seed_groups == artifact_groups, {
    'missing_seed_groups': sorted(artifact_groups - seed_groups),
    'unexpected_seed_groups': sorted(seed_groups - artifact_groups),
}
assert seed_manifest['model_version'] == f'shared_lstm_{RUN_MODE}_v1'
assert seed_manifest['safety_contract']['shared_model_weights'] == 'immutable'
assert seed_manifest['safety_contract']['golden_profiles'] == 'immutable'

archive_base = DRIVE_PROJECT_DIR / 'artifacts' / f'ht9046mx_adaptive_runtime_{RUN_MODE}'
archive_path = Path(shutil.make_archive(
    str(archive_base),
    'zip',
    root_dir=ARTIFACT_DIR.parent,
    base_dir=ARTIFACT_DIR.name,
))

print('Adaptive groups:', len(seed_groups))
print('Runtime ZIP:', archive_path)
print('ZIP size (MB):', round(archive_path.stat().st_size / 1024**2, 1))
print('Safety contract:', seed_manifest['safety_contract'])

if IN_COLAB and not USE_DRIVE:
    from google.colab import files
    print('Downloading trained model and adaptive runtime to the local computer...')
    files.download(str(archive_path))

## Run continuously on Windows

หลัง cell ด้านบนผ่านครบ:

1. รับ `ht9046mx_adaptive_runtime_full.zip` จาก automatic download (`USE_DRIVE=False`) หรือ `MyDrive/Data Analysis/artifacts` (`USE_DRIVE=True`)
2. แตก ZIP ให้ได้โฟลเดอร์ `artifacts/shared_lstm_colab_full` ภายในโปรเจกต์
3. รัน `scripts\initialize_adaptive_runtime.ps1`
4. ทดสอบหนึ่งรอบด้วย `scripts\run_adaptive_cycle.ps1`
5. เมื่อตรวจผล smoke แล้วจึงรัน `scripts\install_adaptive_task.ps1` เพื่อตั้งเวลา Automatic Scoring

Full model นี้ยังเป็น unsupervised anomaly detector ก่อนใช้กำหนด maintenance action ต้องเชื่อมผลกับ maintenance/fault labels และทำ production acceptance test